# Comparing Political Speeches with BERT Embeddings

This notebook walks through a complete pipeline for comparing the semantic content
of political speeches using contextual embeddings from a transformer model:

1. **Load & preprocess** the speeches (RTF → plain text → sentences)
2. **Access RoBERTa** — the *Robustly Optimized BERT* model — via Hugging Face
3. **Generate embeddings** for every sentence, and pool them into document embeddings
4. **PCA** to visualize the high-dimensional embedding space in 2-D
5. **Cosine similarity** to quantify how semantically close the speeches are

**The corpus.** Five U.S. inaugural addresses: Obama 2009, Obama 2013, Trump 2017,
Biden 2021, and Trump 2025. The pipeline works for any number of speeches ≥ 2 —
to compare just two, simply comment out the others in the `SPEECH_FILES`
dictionary below.

**Why transformers instead of bag-of-words?** Classical approaches (TF-IDF, LDA)
treat words as interchangeable tokens: *"bank"* in *"river bank"* and *"bank
loan"* get the same representation. BERT-family models produce **contextual**
embeddings — each word's vector depends on the sentence it appears in — so two
speeches can be judged similar because they *mean* similar things, not merely
because they share vocabulary.

## 0. Setup

We need four libraries beyond the scientific-Python stack:

- `transformers` — Hugging Face's interface to pretrained language models
- `torch` — the deep-learning backend that runs the model
- `striprtf` — converts the `.rtf` speech files to plain text
- `nltk` — for sentence segmentation

On Google Colab, `torch` and `transformers` are pre-installed, so this cell
mostly just adds `striprtf`. A GPU is *not* required — five speeches embed in
under a minute on CPU — but if you have one (Colab: *Runtime → Change runtime
type → GPU*), the code will use it automatically.

In [ ]:
%pip -q install transformers torch striprtf nltk scikit-learn matplotlib

In [ ]:
import re
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

from striprtf.striprtf import rtf_to_text
from transformers import AutoTokenizer, AutoModel
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import nltk
nltk.download("punkt_tab", quiet=True)   # sentence-boundary model
from nltk.tokenize import sent_tokenize

# Reproducibility: PCA has a deterministic solution, but we fix the seed anyway
# in case you extend the notebook with stochastic methods (e.g., t-SNE, UMAP).
np.random.seed(42)

## 1. Load the speeches

The speeches live in `.rtf` files (as exported from a word processor). We map a
human-readable label to each file. **If you run this on Colab**, upload the
files first (folder icon in the left sidebar → upload) and adjust `DATA_DIR` if
needed.

> To compare only two speeches, comment out the lines you don't want — the rest
> of the notebook adapts automatically.

In [ ]:
DATA_DIR = Path(".")   # directory containing the .rtf files

SPEECH_FILES = {
    "Obama 2009":  "obama1.rtf",
    "Obama 2013":  "obama2.rtf",
    "Trump 2017":  "trump1.rtf",
    "Biden 2021":  "biden.rtf",
    "Trump 2025":  "trump2.rtf",
}


def load_text(path):
    """Read a file; if it is RTF, strip the markup and return plain text."""
    raw = Path(path).read_text(encoding="utf-8", errors="ignore")
    if raw.lstrip().startswith("{\\rtf"):          # RTF files start with {\rtf1...
        return rtf_to_text(raw)
    return raw                                     # already plain text


speeches_raw = {label: load_text(DATA_DIR / fname)
                for label, fname in SPEECH_FILES.items()}

for label, text in speeches_raw.items():
    print(f"{label:<12} {len(text):>7,} characters")

## 2. Preprocessing

**Less is more.** With bag-of-words methods we would lowercase, remove stopwords,
and stem. With BERT-family models we do **none** of that:

- The model was pretrained on natural running text — capitalization, function
  words, and punctuation all carry signal it knows how to use.
- Its tokenizer handles out-of-vocabulary words by splitting them into subword
  pieces, so no vocabulary pruning is needed.

What we *do* need:

1. **Remove transcript artifacts** — stage directions like `[Applause]` or
   `[At this point, a moment of silence was observed.]` are the transcriber's
   voice, not the speaker's.
2. **Normalize whitespace** left over from the RTF conversion.
3. **Segment into sentences.** RoBERTa accepts at most 512 subword tokens, and a
   full speech is far longer. The standard solution is to embed each **sentence**
   separately and aggregate afterwards. Sentences are also the natural unit of
   meaning for a speech, which makes the PCA plot below interpretable.

In [ ]:
def preprocess(text):
    """Clean transcript text and split it into sentences."""
    # Collapse whitespace FIRST: in the RTF, stage directions can span lines
    # (e.g., "[\nApplause\n]"), and we want the bracket regex to catch them.
    text = re.sub(r"\s+", " ", text).strip()
    text = re.sub(r"\[.*?\]", " ", text)      # drop [Applause], [Laughter], ...
    sentences = sent_tokenize(text)
    # Guard against fragments (e.g., a stray "Amen." is fine, but "]" is not).
    return [s.strip() for s in sentences if len(s.strip()) > 1]


speeches = {label: preprocess(text) for label, text in speeches_raw.items()}

for label, sents in speeches.items():
    print(f"{label:<12} {len(sents):>4} sentences")

print("\nExample sentences (Obama 2009):")
for s in speeches["Obama 2009"][:3]:
    print("  •", s)

## 3. Load RoBERTa from Hugging Face

We use **`roberta-base`** — *RoBERTa: A Robustly Optimized BERT Pretraining
Approach* (Liu et al., 2019). RoBERTa keeps BERT's architecture (12 transformer
layers, 768-dimensional hidden states, ~125M parameters) but improves the
training recipe: more data (160 GB vs. 16 GB), longer training, dynamic masking,
and no next-sentence-prediction objective. It consistently outperforms the
original BERT on downstream benchmarks.

Two objects come from the Hugging Face Hub:

- **Tokenizer** — turns a sentence into subword IDs the model understands
  (`"unalienable"` → `["un", "alien", "able"]`-style pieces).
- **Model** — the pretrained network. `AutoModel` gives us the *encoder only*
  (no classification head), which is exactly what we want: its hidden states
  *are* the embeddings.

The first run downloads ~500 MB of weights; they are cached afterwards.

In [ ]:
MODEL_NAME = "roberta-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval()   # inference mode: disables dropout

print(f"Hidden size: {model.config.hidden_size} dimensions")

**A peek at tokenization.** Note the `Ġ` symbol — RoBERTa's byte-pair-encoding
marks tokens that begin a new word. Rare words are split into subword pieces, so
the model never encounters an unknown word.

In [ ]:
example = "We hold these truths to be self-evident, that all men are created equal."
tokens = tokenizer.tokenize(example)
print(tokens)

## 4. Generate embeddings

For each sentence, the model returns one 768-dimensional vector **per token**.
We need one vector per *sentence*, so we aggregate — this is called **pooling**.
Two common choices:

- **CLS pooling** — take the vector of the special `<s>` token. Works best when
  a model has been fine-tuned to concentrate sentence meaning there; for an
  off-the-shelf encoder it is often mediocre.
- **Mean pooling** — average the token vectors, ignoring padding. For pretrained
  (not fine-tuned) models this is the standard, more robust choice, and it is
  what sentence-embedding libraries (e.g., Sentence-Transformers) use.

We use **mean pooling with an attention mask**: sentences in a batch are padded
to equal length, and the mask ensures padding tokens do not contaminate the
average.

Finally, the **document embedding** for a speech is the mean of its sentence
embeddings — the "average meaning" of the speech in embedding space.

In [ ]:
@torch.no_grad()                     # no gradients: inference only, saves memory
def embed_sentences(sentences, batch_size=32):
    """Return an (n_sentences, 768) array of mean-pooled RoBERTa embeddings."""
    all_embeddings = []
    for start in range(0, len(sentences), batch_size):
        batch = sentences[start:start + batch_size]
        encoded = tokenizer(batch,
                            padding=True,          # pad batch to equal length
                            truncation=True,       # safety net for long sentences
                            max_length=512,
                            return_tensors="pt").to(device)
        hidden = model(**encoded).last_hidden_state          # (batch, tokens, 768)

        # Mean pooling: zero out padding, sum, divide by true token counts.
        mask = encoded["attention_mask"].unsqueeze(-1)       # (batch, tokens, 1)
        summed = (hidden * mask).sum(dim=1)                  # (batch, 768)
        counts = mask.sum(dim=1).clamp(min=1)                # (batch, 1)
        all_embeddings.append((summed / counts).cpu().numpy())
    return np.vstack(all_embeddings)


# Sentence-level embeddings, kept per speech for the PCA plot
sentence_embeddings = {label: embed_sentences(sents)
                       for label, sents in speeches.items()}

# Document-level embeddings: mean of each speech's sentence embeddings
doc_embeddings = np.vstack([emb.mean(axis=0)
                            for emb in sentence_embeddings.values()])
labels = list(sentence_embeddings.keys())

for label, emb in sentence_embeddings.items():
    print(f"{label:<12} sentence matrix: {emb.shape}")
print(f"\nDocument matrix: {doc_embeddings.shape}   (speeches × dimensions)")

## 5. PCA: visualizing the embedding space

Each sentence now lives in a 768-dimensional space we cannot look at directly.
**Principal Component Analysis** finds the orthogonal directions of greatest
variance and lets us project onto the top two — a "shadow" of the embedding
cloud that preserves as much spread as possible.

Two things to read off the plot:

- **Overlap vs. separation of the clouds.** Inaugural addresses share a genre
  (unity, renewal, God, America), so substantial overlap is expected; what is
  interesting is *which* speeches drift apart.
- **Explained variance.** With only 2 of 768 components we typically capture a
  modest share of the variance — the plot is a legitimate but *lossy* summary,
  which is why we compute cosine similarity in the full space afterwards.

We fit the PCA on **all sentences pooled together** (so every speech is
projected into the same space) and then overlay the projected document
embeddings as large markers.

In [ ]:
# Stack all sentence embeddings; remember which speech each row belongs to
X = np.vstack(list(sentence_embeddings.values()))
speech_of_row = np.concatenate([[label] * len(emb)
                                for label, emb in sentence_embeddings.items()])

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)                     # fit on sentences...
docs_2d = pca.transform(doc_embeddings)         # ...then project the documents

evr = pca.explained_variance_ratio_
print(f"Explained variance: PC1 {evr[0]:.1%}, PC2 {evr[1]:.1%} "
      f"(total {evr.sum():.1%})")

In [ ]:
# One fixed, colorblind-safe color per speech (Okabe–Ito palette)
PALETTE = ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7"]
colors = {label: PALETTE[i % len(PALETTE)] for i, label in enumerate(labels)}

fig, ax = plt.subplots(figsize=(9, 6.5))

for label in labels:
    rows = speech_of_row == label
    ax.scatter(X_2d[rows, 0], X_2d[rows, 1],
               s=22, alpha=0.45, color=colors[label], label=label,
               edgecolors="none")

# Document embeddings: large, outlined, directly labeled
for i, label in enumerate(labels):
    ax.scatter(*docs_2d[i], s=260, marker="*", color=colors[label],
               edgecolors="black", linewidths=0.8, zorder=5)
    ax.annotate(label, docs_2d[i], xytext=(8, 6),
                textcoords="offset points", fontsize=10, fontweight="bold")

ax.set_xlabel(f"PC1 ({evr[0]:.1%} of variance)")
ax.set_ylabel(f"PC2 ({evr[1]:.1%} of variance)")
ax.set_title("Sentence embeddings (dots) and document embeddings (stars), PCA projection")
ax.legend(title="Speech", frameon=False, loc="best")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(True, linewidth=0.4, alpha=0.35)
plt.tight_layout()
plt.show()

**How to read this.** Each faint dot is one sentence; each star is a whole
speech (the mean of its sentences). Distances between stars in this projection
preview the similarity structure — but remember the projection keeps only a
fraction of the variance, so the definitive comparison uses the full
768-dimensional vectors, next.

## 6. Cosine similarity

**Cosine similarity** measures the angle between two vectors:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\lVert\mathbf{a}\rVert \, \lVert\mathbf{b}\rVert}$$

It ranges from −1 (opposite) through 0 (orthogonal) to 1 (identical direction).
Because it ignores vector *length*, it compares the *direction* of meaning —
a long and a short speech about the same themes can still score high.

**A caveat worth teaching:** raw BERT/RoBERTa embeddings are *anisotropic* —
they occupy a narrow cone of the embedding space, so *all* pairwise similarities
tend to be high (often > 0.9). The absolute numbers are therefore hard to
interpret on their own; the **relative ordering** is what carries information.
(If you need calibrated absolute similarities, fine-tuned sentence encoders like
`sentence-transformers/all-mpnet-base-v2` are the tool of choice.)

In [ ]:
sim = cosine_similarity(doc_embeddings)     # (n_speeches, n_speeches)

fig, ax = plt.subplots(figsize=(7, 5.8))
im = ax.imshow(sim, cmap="Blues",
               vmin=sim[~np.eye(len(labels), dtype=bool)].min() - 0.005,
               vmax=1.0)

ax.set_xticks(range(len(labels)), labels, rotation=45, ha="right")
ax.set_yticks(range(len(labels)), labels)

# Direct labels on every cell — the exact values are the point of this figure
for i in range(len(labels)):
    for j in range(len(labels)):
        dark_cell = (sim[i, j] - im.get_clim()[0]) / np.ptp(im.get_clim()) > 0.6
        ax.text(j, i, f"{sim[i, j]:.3f}", ha="center", va="center",
                fontsize=9, color="white" if dark_cell else "#1a1a1a")

ax.set_title("Cosine similarity between document embeddings")
fig.colorbar(im, ax=ax, shrink=0.85, label="cosine similarity")
plt.tight_layout()
plt.show()

In [ ]:
# Rank the pairs from most to least similar
print("Speech pairs, most similar first:\n")
pairs = [(sim[i, j], labels[i], labels[j])
         for i in range(len(labels)) for j in range(i + 1, len(labels))]
for s, a, b in sorted(pairs, reverse=True):
    print(f"  {a:<12} ↔  {b:<12} {s:.4f}")

## 7. Interpretation and where to go next

**What to look for in the results:**

- Do the two speeches by the **same speaker** (Obama 2009/2013, Trump 2017/2025)
  score higher with each other than across speakers? That would suggest the
  embeddings capture a stable individual style/agenda.
- Does **Biden 2021** sit closer to the Obama speeches than to the Trump
  speeches? Substantively we might expect co-partisans to share themes.
- Remember the **anisotropy caveat**: read the *ordering* of the similarities,
  not their absolute magnitudes.

**Natural extensions:**

1. **Fine-tuned sentence encoders.** Swap `roberta-base` for
   `sentence-transformers/all-mpnet-base-v2` (via the `sentence-transformers`
   package) and compare — similarities become better calibrated and the
   ranking often sharpens.
2. **Sentence-level comparison.** Instead of comparing document means, compute
   cosine similarity between all *sentence* pairs across two speeches and
   inspect the most similar pairs — which passages echo each other?
3. **Beyond PCA.** Try UMAP or t-SNE on the sentence embeddings; they often
   reveal thematic clusters (economy, security, unity) that PCA's linear
   projection blurs.
4. **Statistical comparison.** Bootstrap sentences within each speech to put a
   confidence interval on each pairwise similarity.